# PART 9. 고객 이탈 예측 (RandomForest 분류)

## 수업 목표
- 고객의 구매 이력을 요약하고 이탈 가능성을 예측합니다.
- 분류 모델 평가 지표(Accuracy, Precision, Recall, F1)를 이해합니다.

## 수업 진행 포인트
| 구분 | 설명 |
|---|---|
| 데이터 관점 | 고객별로 데이터를 요약(groupby agg)합니다. |
| 코드 관점 | RandomForestClassifier, classification_report 사용법을 익힙니다. |
| AI 관점 | 이탈 고객을 미리 알면 쿠폰/이벤트로 대응할 수 있습니다. |
| 강사 메모 | 이 이탈 라벨은 수업용입니다. 실제 이탈 정답이 아님을 꼭 설명하세요. |

---


## 📌 이 파트에서 사용하는 API 흐름 (scikit-learn)

| 단계 | 함수 | 역할 |
|---|---|---|
| ① | fit() | 모델 학습 |
| ② | predict() | 모델 예측 |
| ③ | score() | 평가 (Accuracy / Precision / Recall / F1) |


## 이탈 기준 (수업용 정의)

| 조건 | 내용 |
|---|---|
| 주문 횟수 | 3회 이하 |
| 마지막 구매 경과일 | 90일 이상 |

두 조건을 모두 만족하면 이탈(1), 아니면 유지(0)

> 실제 서비스에서는 방문 기록, 장바구니, 클릭 데이터 등이 추가로 필요합니다.

In [1]:
# 셀 1. Google Drive 연결

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 셀 2. 라이브러리 불러오기

import os
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

base_path  = "/content/drive/MyDrive/Olist"
model_path = f"{base_path}/model"
os.makedirs(model_path, exist_ok=True)

In [3]:
# 셀 3. 데이터 불러오기

df = pd.read_csv(f"{base_path}/data/olist_master_data.csv")

df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"], errors="coerce"
)

print("데이터 크기:", df.shape)

데이터 크기: (119143, 40)


In [4]:
# 셀 4. 고객별 요약 데이터 생성
# 여러 행에 흩어진 고객 정보를 1명 = 1행으로 요약합니다.

customer_df = df.groupby("customer_unique_id").agg(
    order_count=    ("order_id",               "nunique"),  # 총 주문 횟수
    total_payment=  ("payment_value",           "sum"),      # 총 결제 금액
    avg_review=     ("review_score",            "mean"),     # 평균 리뷰 점수
    last_order_date=("order_purchase_timestamp","max"),      # 마지막 주문일
).reset_index()

print(f"고객 수: {len(customer_df):,}")
customer_df.head()

고객 수: 96,096


,customer_unique_id,order_count,total_payment,avg_review,last_order_date
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,5.0,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,4.0,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,1,86.22,3.0,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,4.0,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,5.0,2017-11-14 19:45:42


In [5]:
# 셀 5. 이탈 라벨 생성
# 데이터 마지막 시점을 2018년 9월 1일로 설정합니다.

today = pd.Timestamp("2018-09-01")

# 마지막 주문 이후 경과일 계산
customer_df["days_since_last_order"] = (
    today - customer_df["last_order_date"]
).dt.days

# 이탈 기준: 주문 3회 이하 AND 90일 이상 미구매
customer_df["churn"] = customer_df.apply(
    lambda row:
        1 if (row["order_count"] <= 3 and row["days_since_last_order"] >= 90)
        else 0,
    axis=1
)

print("=== 이탈 고객 비율 ===")
print(customer_df["churn"].value_counts())
print(customer_df["churn"].value_counts(normalize=True).round(2))

=== 이탈 고객 비율 ===
churn
1    77656
0    18440
Name: count, dtype: int64
churn
1    0.81
0    0.19
Name: proportion, dtype: float64


In [6]:
# 셀 6. 결측치 처리
# 리뷰를 남기지 않은 고객은 avg_review가 NaN

customer_df["avg_review"]    = customer_df["avg_review"].fillna(customer_df["avg_review"].mean())
customer_df["total_payment"] = customer_df["total_payment"].fillna(0)

print("결측치 처리 완료!")

결측치 처리 완료!


In [7]:
# 셀 7. Feature(X) / Target(y) 분리

X = customer_df[[
    "order_count",           # 구매 빈도
    "total_payment",         # 총 구매 금액
    "avg_review",            # 고객 만족도
    "days_since_last_order", # 마지막 구매 이후 경과일
]]
y = customer_df["churn"]

print(f"X 크기: {X.shape}, y 크기: {y.shape}")

X 크기: (96096, 4), y 크기: (96096,)


In [8]:
# 셀 8. train / test 분리

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # 이탈/유지 비율 유지
)

print(f"학습 데이터: {X_train.shape}")
print(f"테스트 데이터: {X_test.shape}")

학습 데이터: (76876, 4)
테스트 데이터: (19220, 4)



### 🔵 ① fit() → 모델 학습


In [10]:
# 셀 9. RandomForest 분류 모델 학습 (약 30초)
# n_estimators=100: 결정 트리 100개를 만들어 다수결로 예측

print("모델 학습 중... (약 30초)")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print("학습 완료!")

모델 학습 중... (약 30초)
학습 완료!



### 🔵 ② predict() → 모델 예측  |  ③ score() → 평가 (Accuracy / Precision / Recall / F1)


In [11]:
# 셀 10. 예측 및 평가

y_pred = model.predict(X_test)

print(f"Accuracy (정확도): {accuracy_score(y_test, y_pred):.4f}")
print()
print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=["유지(0)", "이탈(1)"]))
print("=== Confusion Matrix ===")
cm = confusion_matrix(y_test, y_pred)
print(cm)
print()
print("  대각선 값이 클수록 잘 예측한 것입니다.")
print("  (왼쪽 위 = 유지 맞춤, 오른쪽 아래 = 이탈 맞춤)")

Accuracy (정확도): 1.0000

=== Classification Report ===
              precision    recall  f1-score   support

       유지(0)       1.00      1.00      1.00      3688
       이탈(1)       1.00      1.00      1.00     15532

    accuracy                           1.00     19220
   macro avg       1.00      1.00      1.00     19220
weighted avg       1.00      1.00      1.00     19220

=== Confusion Matrix ===
[[ 3688     0]
 [    0 15532]]

  대각선 값이 클수록 잘 예측한 것입니다.
  (왼쪽 위 = 유지 맞춤, 오른쪽 아래 = 이탈 맞춤)


## 평가 지표 설명

| 지표 | 의미 | 고객 이탈에서 중요한 이유 |
|---|---|---|
| Precision | 이탈이라고 예측한 것 중 진짜 이탈 비율 | 쿠폰 낭비 방지 |
| **Recall** | **실제 이탈 고객을 얼마나 잘 찾았는가** | **이탈 고객을 놓치면 손해** |
| F1-score | Precision + Recall 균형 점수 | 두 지표의 균형 |

> 고객 이탈 예측에서는 **Recall**이 특히 중요합니다.
> 실제 이탈 고객을 놓치면 쿠폰 제공 기회를 잃게 됩니다.

In [12]:
# 셀 11. Feature Importance 확인

importance_df = pd.DataFrame({
    "feature":    X.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("=== Feature 중요도 ===")
print(importance_df)
# ▶ total_payment가 압도적으로 높으면 데이터 구조상 당연한 결과일 수 있습니다.
# ▶ AI 결과를 무조건 신뢰하지 말고, 항상 데이터 구조를 함께 확인하세요.

=== Feature 중요도 ===
                 feature  importance
3  days_since_last_order    0.995778
0            order_count    0.002164
2             avg_review    0.001812
1          total_payment    0.000246


In [13]:
# 셀 12. 모델 저장

joblib.dump(model, f"{model_path}/part9_churn_model.pkl")
print("이탈 예측 모델 저장 완료!")
print("저장 위치:", f"{model_path}/part9_churn_model.pkl")

이탈 예측 모델 저장 완료!
저장 위치: /content/drive/MyDrive/Olist/model/part9_churn_model.pkl




## 마무리 정리

| 확인할 내용 | 설명 |
|---|---|
| 이 파트의 핵심 | 고객 구매 이력으로 이탈 가능성을 예측합니다. |
| 이탈 라벨 주의 | 이 라벨은 수업용이며 실제 이탈 정답이 아닙니다. |
| Recall 중요 | 이탈 고객을 놓치지 않는 것이 핵심입니다. |
| 다음 단계 | PART 10에서 AI 결과를 비즈니스 전략으로 연결합니다. |